# Phase 1 — Lesson 22: Stochastic Processes
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 22: Stochastic Processes
**Type:** Learn | **Languages:** Python | **Time:** ~75 minutes
**Prerequisites:** Phase 1, Lessons 06-07 (Probability, Bayes)



# What You'll Learn

---

## Learning Objectives

- Simulate 1D and 2D random walks and verify the sqrt(n) scaling of displacement
- Build a Markov chain simulator and compute its stationary distribution via eigendecomposition
- Implement Metropolis-Hastings MCMC and Langevin dynamics for sampling from target distributions
- Connect the forward diffusion process to Brownian motion and explain how the reverse process generates data


# The Problem

---

Many AI systems involve randomness that evolves over time:
- Language models generate tokens one at a time (Markov chain)
- Diffusion models add noise step by step (Brownian motion)
- RL agents act in random environments (Markov decision process)
- MCMC samples from posteriors (Markov chain)


# Random Walks

---

At each step, move +1 or −1 with equal probability. Position after n steps:

```
Sₙ = X₁ + X₂ + ... + Xₙ
```

E[Sₙ] = 0 (unbiased), but Var(Sₙ) = n — typical displacement is √n.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)def random_walk_1d(n_steps):    steps = np.random.choice([-1, 1], size=n_steps)    return np.concatenate([[0], np.cumsum(steps)])# Verify sqrt(n) scalingfor n in [100, 400, 900, 1600, 2500]:    walks = np.array([random_walk_1d(n)[-1] for _ in range(5000)])    rms = np.sqrt(np.mean(walks ** 2))    print(f"n={n:5d}: RMS={rms:.2f}, sqrt(n)={np.sqrt(n):.2f}, ratio={rms/np.sqrt(n):.3f}")# Plot several walksfig, axes = plt.subplots(1, 2, figsize=(12, 4))for i in range(5):    axes[0].plot(random_walk_1d(1000), alpha=0.6, lw=1)axes[0].set_title('1D Random Walks', fontweight='bold')axes[0].set_xlabel('Step'); axes[0].set_ylabel('Position')axes[0].grid(True, alpha=0.3)walks_2d = []for i in range(5):    steps = np.random.choice([-1, 1], size=(1000, 2))    pos = np.concatenate([[np.zeros(2)], np.cumsum(steps, axis=0)])    walks_2d.append(pos)    axes[1].plot(pos[:, 0], pos[:, 1], alpha=0.6, lw=1)axes[1].set_title('2D Random Walks', fontweight='bold')axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')axes[1].grid(True, alpha=0.3); axes[1].set_aspect('equal')plt.tight_layout(); plt.show()

# Markov Chains

---

A Markov chain transitions between states with fixed probabilities. The next state depends only on the current state.

```
P[i][j] = P(next = j | current = i)
```

The stationary distribution π satisfies π = π·P — the left eigenvector with eigenvalue 1.


In [ ]:
import numpy as np# Weather Markov chain: Sunny(0), Cloudy(1), Rainy(2)P = np.array([[0.7, 0.2, 0.1],                [0.3, 0.4, 0.3],                [0.2, 0.3, 0.5]])# Power method: multiply distribution by P repeatedlydist = np.array([1.0, 0.0, 0.0])for i in range(50):    dist = dist @ Pprint(f"Stationary distribution: {np.round(dist, 4)}")print(f"(Sunny: {dist[0]:.1%}, Cloudy: {dist[1]:.1%}, Rainy: {dist[2]:.1%})")# Verify: eigenvector of P.T with eigenvalue 1eigenvalues, eigenvectors = np.linalg.eig(P.T)idx = np.argmax(np.abs(eigenvalues))stat = np.real(eigenvectors[:, idx])stat = stat / stat.sum()print(f"\nEigenvector method: {np.round(np.abs(stat), 4)}")

In [ ]:
import matplotlib.pyplot as pltimport numpy as np# Simulate the chain and show convergencedef simulate_chain(P, start, n_steps=1000, seed=42):    rng = np.random.RandomState(seed)    state = start    states = [state]    for _ in range(n_steps):        state = rng.choice(len(P), p=P[state])        states.append(state)    return np.array(states)states = simulate_chain(P, start=0, n_steps=5000)# Empirical distributionempirical = np.bincount(states, minlength=3) / len(states)print(f"Empirical: {np.round(empirical, 4)}")plt.figure(figsize=(7, 4))states_names = ['Sunny', 'Cloudy', 'Rainy']x = np.arange(3)plt.bar(x - 0.2, dist, width=0.4, label='Theoretical', alpha=0.7)plt.bar(x + 0.2, empirical, width=0.4, label='Empirical (5000 steps)', alpha=0.7)plt.xticks(x, states_names)plt.ylabel('Probability')plt.title('Markov Chain Convergence to Stationary Distribution', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3, axis='y')plt.tight_layout(); plt.show()

# Langevin Dynamics

---

Gradient descent with noise:

```
x_{t+1} = x_t − dt·∇U(x_t) + √(2·T·dt)·z_t
```

Gradient pulls toward low energy; noise enables exploration. At equilibrium, samples ∝ exp(−U/T).


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef langevin_dynamics(grad_U, x0, dt, temperature, n_steps, seed=42):    rng = np.random.RandomState(seed)    x = np.array(x0, dtype=float)    traj = [x.copy()]    for _ in range(n_steps):        noise = rng.randn(*x.shape)        x = x - dt * grad_U(x) + np.sqrt(2 * temperature * dt) * noise        traj.append(x.copy())    return np.array(traj)# Double-well potential: U(x) = (x² − 1)²def grad_U(x):    return 4 * x * (x**2 - 1)traj = langevin_dynamics(grad_U, x0=2.0, dt=0.01, temperature=0.1, n_steps=5000)plt.figure(figsize=(7, 4))plt.plot(traj, alpha=0.6, color='#2c3e50')plt.axhline(1, color='green', linestyle='--', label='Well at x=1')plt.axhline(-1, color='green', linestyle='--', label='Well at x=-1')plt.title('Langevin Dynamics in Double-Well Potential', fontsize=12, fontweight='bold')plt.xlabel('Step'); plt.ylabel('Position x')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()# Stationary distribution: histogram of last 3000 stepsplt.figure(figsize=(7, 4))plt.hist(traj[3000:], bins=50, density=True, alpha=0.6, color='#e74c3c', label='Samples')xs = np.linspace(-1.5, 1.5, 200)target = np.exp(-((xs**2 - 1)**2) / 0.1)target = target / target.sum() * (xs[1] - xs[0]) * 5000plt.plot(xs, target, 'k-', lw=2, label='Target ∝ exp(−U/T)')plt.title('Stationary Distribution ∝ exp(−U/T)', fontsize=12, fontweight='bold')plt.xlabel('x'); plt.ylabel('Density')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Metropolis-Hastings MCMC

---

Sample from p(x) using a proposal Q:

1. Propose x′ ~ Q(x′\|x)
2. Accept with probability min(1, p(x′)/p(x)) (symmetric Q)
3. Repeat — chain converges to p(x)


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef metropolis_hastings(target_log_prob, x0, n_samples=10000, proposal_std=0.5, seed=42):    rng = np.random.RandomState(seed)    x = x0    samples = [x]    for _ in range(n_samples):        x_prop = x + rng.randn() * proposal_std        log_ratio = target_log_prob(x_prop) - target_log_prob(x)        if np.log(rng.rand()) < log_ratio:            x = x_prop        samples.append(x)    return np.array(samples)# Target: bimodal distributiondef target_log_prob(x):    return np.log(0.5 * np.exp(-(x - 2)**2 / 2) + 0.5 * np.exp(-(x + 2)**2 / 2))samples = metropolis_hastings(target_log_prob, x0=0.0, n_samples=20000)plt.figure(figsize=(7, 4))plt.hist(samples, bins=50, density=True, alpha=0.6, color='#2c3e50', label='MH samples')xs = np.linspace(-4, 4, 200)target = 0.5 * np.exp(-(xs - 2)**2 / 2) + 0.5 * np.exp(-(xs + 2)**2 / 2)plt.plot(xs, target, 'k-', lw=2, label='Target PDF')plt.title('Metropolis-Hastings: Bimodal Target', fontsize=12, fontweight='bold')plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()print(f"Samples mean: {np.mean(samples):.4f} (should be ~0)")print(f"Samples std:  {np.std(samples):.4f}")

# Connection to Diffusion Models

---

Diffusion models add noise step by step (forward Markov chain), then learn to denoise (reverse chain).

Forward: xₜ = √(1−βₜ)·xₜ₋₁ + √βₜ·ε — gradually becomes pure noise.
Reverse: learned neural network predicts and removes noise step by step.


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Forward diffusion: add noise progressivelynp.random.seed(42)x0 = np.sin(np.linspace(0, 4 * np.pi, 100))  # Clean signalT = 50betas = np.linspace(0.0001, 0.02, T)alphas = 1 - betasalphas_bar = np.cumprod(alphas)noisy_signals = [x0]for t in range(T):    noise = np.random.randn(*x0.shape)    x_t = np.sqrt(alphas_bar[t]) * x0 + np.sqrt(1 - alphas_bar[t]) * noise    noisy_signals.append(x_t)fig, axes = plt.subplots(1, 4, figsize=(12, 3))for ax, t_idx, label in [(axes[0], 0, 't=0 (clean)'), (axes[1], 10, 't=10'), (axes[2], 25, 't=25'), (axes[3], 49, 't=49 (noisy)')]:    ax.plot(noisy_signals[t_idx], color='#2c3e50', lw=1.5)    ax.set_title(label, fontweight='bold')    ax.set_xticks([]); ax.set_yticks([])    ax.set_ylim(-3, 3)plt.suptitle('Forward Diffusion: Signal → Pure Noise', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()print("Forward diffusion gradually adds noise until signal becomes pure Gaussian noise.")print("The reverse process (learned by a neural network) removes noise step by step.")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What is the √n scaling in random walks?
> **Explanation:** Typical displacement grows as √n — signature of independent random steps.

**Q2:** What is the stationary distribution?
> **Explanation:** The distribution π where π = π·P — the chain's long-run equilibrium.

**Q3:** What does Metropolis-Hastings do?
> **Explanation:** Constructs a Markov chain whose stationary distribution is the target p(x).



# Practice Exercises

---

Do these in order:

**Exercise 1:** Simulate 1000 random walks of 10000 steps; verify final positions are Gaussian with std √10000 = 100.

**Exercise 2:** Build a text generator using a Markov chain over words; sample new sentences.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Random walk | Coin-flip movement | Cumulative sum of random ±1 steps |
| Markov property | Memoryless | Future depends only on present, not history |
| Stationary distribution | Long-run average | π where π·P = π — chain's equilibrium |
| Brownian motion | Random jiggling | Continuous-time limit of random walk |
| Langevin dynamics | Gradient + noise | Gradient descent with calibrated noise |
| MCMC | Walking to target | Chain whose stationary distribution is p(x) |
| Metropolis-Hastings | Propose and accept | MCMC using acceptance ratios |
| Diffusion process | Noise in, noise out | Forward: add noise. Reverse: remove it (learned) |


# Further Reading

---

- [Ho et al. (2020): Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239)
- [Song & Ermon (2019): Score-Based Generative Modeling](https://arxiv.org/abs/1901.08581)
- [Norris (1997): Markov Chains](https://www.cambridge.org/core/books/markov-chains/)


# Next Steps

---

When you're ready, continue to Phase 2: ML Fundamentals.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
